<a href="https://colab.research.google.com/github/Atharv-1905/Deep-Learning/blob/main/ResNet50(Transfer_Learning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# 1. Install the Kaggle library
!pip install -q kaggle

# 2. Upload the kaggle.json file you just downloaded
from google.colab import files
files.upload()

# 3. Create a kaggle directory and move the file there
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

# 4. Set permissions so the API can read the file
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle (1).json


In [22]:
# Example: Replace this with your copied API command
!kaggle datasets download -d akash2sharma/tiny-imagenet

# Unzip the downloaded file
import zipfile
import os

# Find the name of the zip file (usually the last part of the dataset name)
zip_file = "tiny-imagenet.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('./data') # Extracts into a 'data' folder

print("Dataset unzipped successfully!")

Dataset URL: https://www.kaggle.com/datasets/akash2sharma/tiny-imagenet
License(s): unknown
tiny-imagenet.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset unzipped successfully!


In [23]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

In [24]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 32

In [25]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # Using 20% for validation
)

In [26]:
train_generator = train_datagen.flow_from_directory(
    './data/tiny-imagenet-200/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

Found 80000 images belonging to 200 classes.


In [27]:
val_generator = train_datagen.flow_from_directory(
    './data/tiny-imagenet-200/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 20000 images belonging to 200 classes.


In [28]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(64, 64, 3))

base_model.trainable = False

In [29]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(), # ResNet works better with Global Pooling than Flatten
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(200, activation='softmax') # 200 classes for Tiny ImageNet
])


In [30]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 2, 2, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 200)            │       102,600 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,739,400 (94.37 MB)

 Trainable params: 1,151,688 (4.39 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [31]:
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor the validation loss
    patience=3,          # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored quantity
)

In [33]:
history = model.fit(
    train_generator,       # Your training data
    epochs=1,             # Number of passes through data
    validation_data=val_generator, # Data the model hasn't 'seen'
    callbacks=[early_stopping], # Add the EarlyStopping callback
    verbose=1
)

2500/2500 ━━━━━━━━━━━━━━━━━━━━ 1606s 642ms/step - accuracy: 0.0077 - loss: 5.2877 - val_accuracy: 0.0124 - val_loss: 5.2471


In [34]:
print(f"Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

Training Accuracy: 0.0077
Validation Accuracy: 0.0124
